In [43]:
import numpy as np 
import pandas as pd 
import joblib

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer 
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.preprocessing import OneHotEncoder,StandardScaler

In [12]:
df=pd.read_csv("data\prop__.csv")

In [47]:
df['zone'].unique()

array(['Urban', 'Semi-Urban'], dtype=object)

In [15]:
city_tier_1=['Noida','Dwarka','Gurgaon','Rohini']

In [16]:
def tier_city(location):
    if location in city_tier_1:
        return 1
    else:
        return 2

In [17]:
df['tier_city']=df['location'].apply(tier_city)

In [46]:
df.sample(5)

,area_sqft,price_lakh,location,property_category,property_type,distance_hospital_km,distance_airport_km,zone,tier_city
175,1246,222.72,Gurgaon,Residential,Commercial,4.42,14.12,Semi-Urban,1
52,1416,211.96,Noida,Apartment,3BHK,1.79,27.08,Urban,1
839,3262,421.24,Ghaziabad,Apartment,Villa,6.65,8.65,Urban,2
590,1321,267.17,Vasant Kunj,Residential,3BHK,11.12,16.78,Urban,2
748,1091,170.52,Gurgaon,Residential,3BHK,2.03,25.35,Urban,1


In [23]:
x=df.loc[:,["area_sqft","property_category","property_type","distance_hospital_km","distance_airport_km","zone",'tier_city']]

In [25]:
y=df['price_lakh']

In [31]:
prep=ColumnTransformer(transformers=[('num',"passthrough",['area_sqft','distance_hospital_km','distance_airport_km','tier_city']),
                                     ('cat',OneHotEncoder(handle_unknown='ignore'),['property_category','property_type','zone'])])

In [32]:
x_tr,x_ts,y_tr,y_ts=train_test_split(x,y,test_size=0.2)

In [33]:
model=RandomForestRegressor(n_estimators=300,
    max_depth=16,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1)

In [34]:
pipe=Pipeline(steps=[('prep',prep),
                     ('model',model)])

In [35]:
pipe.fit(x_tr,y_tr)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['area_sqft',
                                                   'distance_hospital_km',
                                                   'distance_airport_km',
                                                   'tier_city']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['property_category',
                                                   'property_type',
                                                   'zone'])])),
                ('model',
                 RandomForestRegressor(max_depth=16, max_features='sqrt',
                                       min_samples_leaf=2, min_samples_split=4,
                                       n_estimators=300, n_jobs=-1,
                                       random_state=42))])

In [37]:
y_pred=pipe.predict(x_ts)

In [39]:
r2_score(y_ts,y_pred)

0.6555985327855791

In [41]:
mean_absolute_error(y_ts,y_pred)

61.82783724902517

In [42]:
mean_squared_error(y_ts,y_pred)

12450.776666705222

In [44]:
joblib.dump(pipe,"prop_model.joblib")

['prop_model.joblib']